# Task 10: NER для соцсетей

Ноутбук повторяет подход из `lab10.ipynb`:
- дообучение `DeepPavlov/rubert-base-cased` для token classification;
- генерация предсказаний в формате списка BIO-тегов;
- сохранение результата в `task-10-predictions.txt`.


In [1]:
# Если пакеты уже стоят, эту ячейку можно пропустить.
%pip install -q transformers datasets seqeval nerus accelerate scikit-learn

Note: you may need to restart the kernel to use updated packages.


In [2]:
import json
import random
import zipfile
from pathlib import Path
from urllib.request import urlretrieve

import numpy as np
import torch
from datasets import Dataset, DatasetDict
from nerus import load_nerus
from seqeval.metrics import accuracy_score, classification_report, f1_score
from sklearn.model_selection import train_test_split
from transformers import (
    AutoModelForTokenClassification,
    AutoTokenizer,
    DataCollatorForTokenClassification,
    Trainer,
    TrainingArguments,
)


/Users/afafos/Desktop/Program/itmo/automatic-text-processing-itmo/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

device

device(type='cpu')

In [4]:
label_list = ["O", "B-LOC", "I-LOC", "B-ORG", "I-ORG", "B-PER", "I-PER"]
label2id = {label: i for i, label in enumerate(label_list)}
id2label = {i: label for i, label in enumerate(label_list)}
label2id

{'O': 0,
 'B-LOC': 1,
 'I-LOC': 2,
 'B-ORG': 3,
 'I-ORG': 4,
 'B-PER': 5,
 'I-PER': 6}

In [5]:
NERUS_URL = "https://storage.yandexcloud.net/natasha-nerus/data/nerus_lenta.conllu.gz"
NERUS_PATH = Path("nerus_lenta.conllu.gz")

if not NERUS_PATH.exists():
    urlretrieve(NERUS_URL, NERUS_PATH)

NERUS_PATH

PosixPath('nerus_lenta.conllu.gz')

In [6]:
# Для ускорения берём часть NERUS, как в референсном ноутбуке.
nerus = load_nerus(str(NERUS_PATH))

sentences = []
tags = []

for doc_id, doc in enumerate(nerus):
    if doc_id >= 2500:
        break
    for sent in doc.sents:
        words = [token.text for token in sent.tokens]
        ner_tags = [token.tag for token in sent.tokens]
        if words:
            sentences.append(words)
            tags.append(ner_tags)

len(sentences), len(tags)

(29586, 29586)

In [7]:
train_tokens, val_tokens, train_tags, val_tags = train_test_split(
    sentences,
    tags,
    test_size=0.1,
    random_state=SEED,
)

dataset = DatasetDict(
    {
        "train": Dataset.from_dict({"tokens": train_tokens, "ner_tags": train_tags}),
        "validation": Dataset.from_dict({"tokens": val_tokens, "ner_tags": val_tags}),
    }
)

dataset

DatasetDict({
    train: Dataset({
        features: ['tokens', 'ner_tags'],
        num_rows: 26627
    })
    validation: Dataset({
        features: ['tokens', 'ner_tags'],
        num_rows: 2959
    })
})

In [8]:
model_name = "DeepPavlov/rubert-base-cased"
tokenizer = AutoTokenizer.from_pretrained(model_name)


def tokenize_and_align_labels(examples):
    tokenized_inputs = tokenizer(
        examples["tokens"],
        truncation=True,
        is_split_into_words=True,
        max_length=128,
    )

    labels = []
    for i, label in enumerate(examples["ner_tags"]):
        word_ids = tokenized_inputs.word_ids(batch_index=i)
        previous_word_idx = None
        label_ids = []

        for word_idx in word_ids:
            if word_idx is None:
                label_ids.append(-100)
            elif word_idx != previous_word_idx:
                label_ids.append(label2id[label[word_idx]])
            else:
                current_label = label[word_idx]
                if current_label.startswith("B-"):
                    current_label = "I-" + current_label[2:]
                label_ids.append(label2id[current_label])
            previous_word_idx = word_idx

        labels.append(label_ids)

    tokenized_inputs["labels"] = labels
    return tokenized_inputs


tokenized_datasets = dataset.map(tokenize_and_align_labels, batched=True)
tokenized_datasets

Map: 100%|██████████| 2959/2959 [00:00<00:00, 34449.49 examples/s]


DatasetDict({
    train: Dataset({
        features: ['tokens', 'ner_tags', 'input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 26627
    })
    validation: Dataset({
        features: ['tokens', 'ner_tags', 'input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 2959
    })
})

In [9]:
model = AutoModelForTokenClassification.from_pretrained(
    model_name,
    num_labels=len(label_list),
    id2label=id2label,
    label2id=label2id,
).to(device)


def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=2)

    true_predictions = [
        [label_list[pred] for pred, lab in zip(prediction, label) if lab != -100]
        for prediction, label in zip(predictions, labels)
    ]

    true_labels = [
        [label_list[lab] for pred, lab in zip(prediction, label) if lab != -100]
        for prediction, label in zip(predictions, labels)
    ]

    return {
        "token_acc": accuracy_score(true_labels, true_predictions),
        "f1": f1_score(true_labels, true_predictions),
    }


training_args = TrainingArguments(
    output_dir="./ner_rubert_task10",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="steps",
    logging_steps=100,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    report_to="none",
    fp16=torch.cuda.is_available(),
    seed=SEED,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    data_collator=DataCollatorForTokenClassification(tokenizer=tokenizer),
    compute_metrics=compute_metrics,
)


Loading weights: 100%|██████████| 197/197 [00:00<00:00, 61483.58it/s]
BertForTokenClassification LOAD REPORT from: DeepPavlov/rubert-base-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
bert.pooler.dense.bias                     | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
bert.pooler.dense.weight                   | UNEXPECTED | 
bert.embeddings.position_ids               | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
classifier.weight             

In [10]:
# Обучение (самая долгая ячейка)
trainer.train()

/Users/afafos/Desktop/Program/itmo/automatic-text-processing-itmo/.venv/lib/python3.13/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss,Token Acc,F1
1,0.025573,0.020257,0.993580,0.959368
2,0.010878,0.022678,0.993564,0.959170
3,0.009940,0.023034,0.994426,0.963271


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  3.22it/s]
/Users/afafos/Desktop/Program/itmo/automatic-text-processing-itmo/.venv/lib/python3.13/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  3.87it/s]
/Users/afafos/Desktop/Program/itmo/automatic-text-processing-itmo/.venv/lib/python3.13/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  3.55it/s]
There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bi

TrainOutput(global_step=4995, training_loss=0.02313184091398069, metrics={'train_runtime': 580.3512, 'train_samples_per_second': 137.643, 'train_steps_per_second': 8.607, 'total_flos': 1853290703529738.0, 'train_loss': 0.02313184091398069, 'epoch': 3.0})

In [11]:
# Проверка качества на валидации
metrics = trainer.predict(tokenized_datasets["validation"]).metrics
metrics

/Users/afafos/Desktop/Program/itmo/automatic-text-processing-itmo/.venv/lib/python3.13/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


{'test_loss': 0.023034121841192245,
 'test_token_acc': 0.9944264742473252,
 'test_f1': 0.9632709632709633,
 'test_runtime': 4.0076,
 'test_samples_per_second': 738.353,
 'test_steps_per_second': 46.163}

In [12]:
predictions, labels, _ = trainer.predict(tokenized_datasets["validation"])
predictions = np.argmax(predictions, axis=2)

true_predictions = [
    [label_list[pred] for pred, lab in zip(prediction, label) if lab != -100]
    for prediction, label in zip(predictions, labels)
]

true_labels = [
    [label_list[lab] for pred, lab in zip(prediction, label) if lab != -100]
    for prediction, label in zip(predictions, labels)
]

print(classification_report(true_labels, true_predictions, digits=4))
print("token_acc =", accuracy_score(true_labels, true_predictions))
print("f1 =", f1_score(true_labels, true_predictions))

              precision    recall  f1-score   support

         LOC     0.9867    0.9828    0.9847      1279
         ORG     0.9138    0.9361    0.9248      1189
         PER     0.9805    0.9796    0.9800      1128

   micro avg     0.9602    0.9664    0.9633      3596
   macro avg     0.9603    0.9662    0.9632      3596
weighted avg     0.9606    0.9664    0.9634      3596

token_acc = 0.9944264742473252
f1 = 0.9632709632709633


In [13]:
TEST_ZIP_URL = "https://storage.yandexcloud.net/lms-itmo-ru-files-27a87tyf/Social_Networks/Task_3/Soc_Net_Task_3_File_0.zip"
TEST_ZIP_PATH = Path("Soc_Net_Task_3_File_0.zip")
TEST_JSON_PATH = Path("Soc_Net_Task_3_File_0.json")

if not TEST_ZIP_PATH.exists():
    urlretrieve(TEST_ZIP_URL, TEST_ZIP_PATH)

if not TEST_JSON_PATH.exists():
    with zipfile.ZipFile(TEST_ZIP_PATH, "r") as zf:
        json_members = [name for name in zf.namelist() if name.endswith(".json")]
        if not json_members:
            raise FileNotFoundError("В архиве не найден JSON-файл с тестом")
        with zf.open(json_members[0]) as src, open(TEST_JSON_PATH, "wb") as dst:
            dst.write(src.read())

with open(TEST_JSON_PATH, "r", encoding="utf-8") as f:
    test_data = json.load(f)

test_sentences = test_data["sentences"]
len(test_sentences), sum(len(s) for s in test_sentences)

(500, 9243)

In [15]:
model.eval()
model_device = next(model.parameters()).device


def predict_sentence_tokens(words):
    inputs = tokenizer(
        words,
        truncation=True,
        is_split_into_words=True,
        max_length=128,
        return_tensors="pt",
    )
    inputs = {k: v.to(model_device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model(**inputs)

    pred_ids = outputs.logits.argmax(-1).squeeze(0).detach().cpu().numpy()
    word_ids = tokenizer(
        words,
        truncation=True,
        is_split_into_words=True,
        max_length=128,
    ).word_ids()

    result = []
    prev_word = None
    for pred_id, word_id in zip(pred_ids, word_ids):
        if word_id is None:
            continue
        if word_id != prev_word:
            result.append(id2label[int(pred_id)])
            prev_word = word_id

    return result


all_predictions = []

for sent in test_sentences:
    pred_tags = predict_sentence_tokens(sent)

    # Защита от редких рассинхронов по длине после токенизации.
    if len(pred_tags) != len(sent):
        pred_tags = pred_tags[: len(sent)]
        if len(pred_tags) < len(sent):
            pred_tags.extend(["O"] * (len(sent) - len(pred_tags)))

    all_predictions.extend(pred_tags)

len(all_predictions), sum(len(s) for s in test_sentences)

(9243, 9243)

In [16]:
print(all_predictions[:30])
print("Predictions count:", len(all_predictions))

['O', 'O', 'B-PER', 'I-PER', 'O', 'O', 'O', 'O', 'O', 'B-LOC', 'O', 'O', 'O', 'O', 'O', 'O', 'B-ORG', 'I-ORG', 'O', 'O', 'O', 'O', 'B-PER', 'O', 'O', 'O', 'O', 'O', 'O', 'O']
Predictions count: 9243


In [17]:
# Формат для Moodle: Python-список строк в верхнем регистре.
submission = str(all_predictions).replace(" ", "")
print(submission)

with open("task-10-predictions.txt", "w", encoding="utf-8") as f:
    f.write(submission)

print("Сохранено в task-10-predictions.txt")

['O','O','B-PER','I-PER','O','O','O','O','O','B-LOC','O','O','O','O','O','O','B-ORG','I-ORG','O','O','O','O','B-PER','O','O','O','O','O','O','O','O','O','O','O','O','O','O','O','O','B-PER','I-PER','O','O','O','O','O','O','O','O','B-PER','I-PER','O','O','O','O','O','O','O','O','O','O','O','O','O','O','O','O','O','O','O','O','O','O','O','B-PER','O','O','O','O','O','O','O','O','O','O','O','O','O','O','O','O','O','O','O','O','O','O','O','O','O','O','O','O','O','O','O','O','O','O','O','O','O','O','O','O','O','O','O','O','O','O','O','O','O','O','O','O','O','O','O','O','O','O','O','O','O','B-PER','O','O','O','O','O','O','O','O','O','O','O','O','O','O','O','O','O','O','O','O','B-LOC','O','B-PER','I-PER','O','O','B-PER','I-PER','O','O','B-PER','I-PER','O','O','O','B-PER','I-PER','O','O','B-ORG','O','O','O','B-ORG','I-ORG','I-ORG','O','O','O','O','O','O','O','O','O','O','O','O','O','O','O','O','O','O','O','O','O','O','O','O','O','O','O','B-PER','O','O','O','O','O','O','O','O','O','O','O','B-LOC'